# 02 — Confirmatory model

**Pre-registered confirmatory spec (2026-10-22, written before running the model)**
- Sample: US-IPO matched set with a day-30 return (n=306). First-day return is missing for 20 firms whose close equals the offer price (n=286).
- Outcomes: `pop_log` = ln(1 + first-day return); `ret30_log` = ln(1 + day-30 return).
- Test variable: `log_pre30` = ln(pre-IPO 30-day views + 1).
- Controls: year fixed effects, `foreign_issuer`, ln(offer price).
- Estimator: OLS with HC3 robust standard errors. Report coef, 95% CI, p, n, R².
- Spec A (main): all firms, plus a `has_pre_ipo_article` dummy (no-article firms count as 0 views).
- Spec B (robustness): exclude no-article firms.
- Interpretation: coef × ln 2 ≈ change in ln(1 + return) per doubling of views (≈ % change in 1 + return).
- Secondary, exploratory: sector heterogeneity per sector with n ≥ 15. Baselines: return descriptives by article group, sector and year (n < 15 suppressed), Kruskal–Wallis across article groups, and a controls-only model to give attention's incremental R².
- Decision rule: the attention coefficient counts as confirmed only if p < 0.05 in Spec A and has the same sign in Spec B. Each outcome is reported separately; there is no multiple-comparison adjustment across the 2 outcomes × 2 specs, and that is stated.


*This spec was written down (here and in `plan.md`) **before** the model ran. Writing it first stops us from trying many versions and reporting the one that looks best.*

## Setup
All outputs below are aggregates: group statistics and regression coefficients. Groups with n < 15 are suppressed. The return data is licensed vendor data, used only as an outcome.

In [ ]:
import numpy as np, pandas as pd, statsmodels.formula.api as smf
from scipy.stats import kruskal
MIN_N = 15
t = pd.read_parquet('../data/processed/analysis_table.parquet')
df = t[t.usable_day30].copy()
df['log_pre30'] = np.log1p(df.views_pre30)
df['log_offer'] = np.log(df.offer_price.astype(float))
df['foreign'] = df.foreign_issuer.astype(int)
df['no_article'] = (~df.has_pre_ipo_article).astype(int)
df['year'] = df.year.astype(str)
df['article_group'] = np.select([~df.has_pre_ipo_article, df.views_pre30 == 0], ['no article', 'article, zero pre-IPO views'], 'article with views')
vc = df.sector.value_counts(); df['sec'] = df.sector.where(df.sector.map(vc) >= MIN_N, 'Other')
print(len(df), df['pop'].notna().sum()); df.article_group.value_counts()

## 1. Baselines: what do IPO returns look like without any model?

Before asking whether attention matters, describe the returns themselves: the average, the typical (median) value, the spread (SD, IQR), and how often they're positive. This shows how big a "normal" IPO return is, so a later effect can be judged against it.

In [ ]:
def desc(g, y):
    x = g[y].dropna()
    return pd.Series({'n': len(x), 'mean': x.mean(), 'median': x.median(), 'sd': x.std(), 'iqr': x.quantile(.75) - x.quantile(.25), 'pct_positive': (x > 0).mean() * 100})
def table(by):
    out = []
    for y, lab in (('pop', 'first-day'), ('ret30', 'day-30')):
        groups = [('All', df)] if by is None else list(df.groupby(by))
        for k, g in groups:
            r = desc(g, y); r['outcome'] = lab; r['group'] = k; out.append(r)
    o = pd.DataFrame(out)[['outcome','group','n','mean','median','sd','iqr','pct_positive']]
    small = o.n < MIN_N
    o[['mean','median','sd','iqr','pct_positive']] = o[['mean','median','sd','iqr','pct_positive']].round(3).astype(object).where(~small, 'suppressed (n<15)')
    return o
display(table(None)); display(table('article_group')); display(table('sec')); display(table('year'))

**Kruskal–Wallis** asks whether the three article groups have the same return distribution. It uses ranks, so the extreme returns can't dominate it. A small p means at least one group differs.

In [ ]:
for y in ('pop', 'ret30'):
    gs = [g[y].dropna() for _, g in df.groupby('article_group')]
    h, p = kruskal(*gs); print(f'{y}: H={h:.2f}, p={p:.4f}, n={sum(map(len, gs))}')

## 2. Confirmatory regressions

OLS fits a straight-line relationship while holding the controls fixed. Year fixed effects absorb "2021 was a hot market". `foreign` and `log_offer` absorb the other strong patterns we found in the EDA. HC3 standard errors stay honest when the spread of errors isn't constant, which is very likely with IPO returns.

`log_pre30` is the coefficient of interest. Multiply it by ln 2 ≈ 0.693 for the effect of **doubling** pre-IPO views on ln(1 + return), which is roughly the % change in (1 + return).

In [ ]:
CTRL = 'C(year) + foreign + log_offer'
def fit(y, data, extra=''):
    d = data.dropna(subset=[y])
    return smf.ols(f'{y} ~ log_pre30 {extra} + {CTRL}', d).fit(cov_type='HC3'), smf.ols(f'{y} ~ {CTRL}' + (f' {extra}' if extra else ''), d).fit(cov_type='HC3')
rows = []; models = {}
for y in ('pop_log', 'ret30_log'):
    for spec, data, extra in (('A (main)', df, '+ no_article'), ('B (robust)', df[df.has_pre_ipo_article], '')):
        m, base = fit(y, data, extra); models[(y, spec)] = m
        ci = m.conf_int().loc['log_pre30']
        rows.append(dict(outcome=y, spec=spec, n=int(m.nobs), coef=m.params.log_pre30, ci_low=ci[0], ci_high=ci[1], p=m.pvalues.log_pre30,
                         per_doubling=m.params.log_pre30*np.log(2), r2=m.rsquared, r2_controls_only=base.rsquared, incr_r2=m.rsquared-base.rsquared,
                         no_article_coef=m.params.get('no_article', np.nan), no_article_p=m.pvalues.get('no_article', np.nan)))
res = pd.DataFrame(rows).round(4); res

Full output for Spec A, first-day, so you can see every coefficient:

In [ ]:
print(models[('pop_log', 'A (main)')].summary())

## 3. Sector heterogeneity — *secondary / exploratory*

Does the attention effect differ by sector? Each sector with n ≥ 15 gets its own Spec A model. Samples are small, so the intervals are wide. These are **not** confirmatory: 6 sectors × 2 outcomes is 12 extra tests, and about one could look significant by chance. These use **HC1** robust SEs, because HC3 breaks down when a sector has a year with a single firm.

In [ ]:
srows = []
for s, g in df.groupby('sec'):
    for y in ('pop_log', 'ret30_log'):
        d = g.dropna(subset=[y])
        if len(d) < MIN_N or s == 'Other': continue
        try:
            m = smf.ols(f'{y} ~ log_pre30 + no_article + C(year) + foreign + log_offer', d).fit(cov_type='HC1')  # HC1: HC3 is undefined when a sector has a year with one firm (leverage = 1)
            ci = m.conf_int().loc['log_pre30']
            srows.append(dict(sector=s, outcome=y, n=len(d), coef=m.params.log_pre30, ci_low=ci[0], ci_high=ci[1], p=m.pvalues.log_pre30, per_doubling=m.params.log_pre30*np.log(2)))
        except Exception as e:
            srows.append(dict(sector=s, outcome=y, n=len(d), note=str(e)[:60]))
pd.DataFrame(srows).round(4)

## 5. Exploratory (post-hoc): market cap

**Not part of the pre-registered spec.** Caleb asked this after seeing the results, so treat everything here as exploratory.

**IPO-time market cap** = offer price × shares outstanding right after the offering, as stated in the 424B4 summary (base deal, before the over-allotment). No vendor prices are used.
- *Multiple share classes:* summed (or the prospectus's own stated total). For Up-C structures we use the "assuming exchange of all units" count when the prospectus gives one.
- *ADS deals:* the share count is in ordinary shares, so it's converted to ADS-equivalents with the ADS ratio before multiplying by the per-ADS offer price.
- Rows the parser couldn't read reliably are left **missing**, not guessed (see coverage below).

Why it matters: bigger companies are better known, so they likely get more Wikipedia views. If size also drives returns, attention could just be standing in for size.

In [ ]:
df = df.merge(t[['cik']], on='cik')
df['log_cap'] = np.log(df.market_cap)
print('market cap available:', df.market_cap.notna().sum(), 'of', len(df))
print(t[t.usable_day30].cap_flags.fillna('').str.split().explode().value_counts().to_string())
print((df.market_cap/1e9).describe().round(2))

In [ ]:
from scipy.stats import spearmanr
rows = []
for inc in (True, False):
    d = df if inc else df[df.has_pre_ipo_article]
    for x, lab in (('views_pre30','pre-IPO 30d views'),('views_day0','IPO-day views'),('views_post30','post-IPO 30d views'),('pop','first-day return'),('ret30','day-30 return')):
        g = d[['log_cap', x]].dropna(); r, p = spearmanr(g.log_cap, g[x])
        rows.append(dict(include_no_article=inc, vs=lab, n=len(g), rho=round(r,3), p=round(p,4)))
pd.DataFrame(rows)

**Does controlling for size change the attention coefficient?** We re-run Spec A and Spec B on the firms with a market cap, first without and then with `log_cap`. Same sample both times, so the only difference is the size control.

In [ ]:
rows = []
for y in ('pop_log','ret30_log'):
    for spec, data, extra in (('A', df, '+ no_article'), ('B', df[df.has_pre_ipo_article], '')):
        d = data.dropna(subset=[y, 'log_cap'])
        for lab, f in (('without log_cap', f'{y} ~ log_pre30 {extra} + {CTRL}'), ('with log_cap', f'{y} ~ log_pre30 {extra} + log_cap + {CTRL}')):
            m = smf.ols(f, d).fit(cov_type='HC3'); ci = m.conf_int().loc['log_pre30']
            rows.append(dict(outcome=y, spec=spec, model=lab, n=int(m.nobs), coef=m.params.log_pre30, ci_low=ci[0], ci_high=ci[1], p=m.pvalues.log_pre30,
                             per_doubling=m.params.log_pre30*np.log(2), cap_coef=m.params.get('log_cap', np.nan), cap_p=m.pvalues.get('log_cap', np.nan), r2=m.rsquared))
pd.DataFrame(rows).round(4)

### Size tiers
micro < \$300M, small \$300M–\$2B, mid \$2B–\$10B, large > \$10B. Medians per tier; any tier with n < 15 is suppressed.

In [ ]:
def tier(c):
    return pd.cut(c/1e9, [0, .3, 2, 10, np.inf], labels=['micro (<$0.3B)','small ($0.3-2B)','mid ($2-10B)','large (>$10B)'], right=False)
out = []
for inc in (True, False):
    d = (df if inc else df[df.has_pre_ipo_article]).dropna(subset=['market_cap']).assign(tier=lambda x: tier(x.market_cap))
    for k, g in d.groupby('tier', observed=True):
        out.append(dict(include_no_article=inc, tier=k, n=len(g), med_pre30=g.views_pre30.median(), med_day0=g.views_day0.median(), med_post30=g.views_post30.median(),
                        med_first_day=g['pop'].median(), n_first_day=g['pop'].notna().sum(), med_day30=g.ret30.median()))
tt = pd.DataFrame(out)
vals = [c for c in tt.columns if c.startswith('med')]
tt[vals] = tt[vals].round(3).astype(object).where(tt.n >= MIN_N, 'suppressed (n<15)')
tt

## 4. Reading the results

See the summary in the project report. The decision rule is in the spec cell at the top: confirmed only if p < 0.05 in Spec A **and** the same sign in Spec B.